# Enhancer-level in silico knockouts

Every enhancer of a target gene, and every combination of them, is replaced by
dinucleotide-shuffled sequence inside one 1 Mb AlphaGenome window. Predicted
target-gene expression and enhancer accessibility are then modelled with a
log-link GLM: main effects $\beta_i$ give the effect of losing one enhancer,
interaction terms $\beta_{ij}$ the deviation of a pair from a multiplicative
null ($\beta_{ij} < 0$ super-additive, $> 0$ sub-additive). The estimates are
compared with the dual CRISPRi screen.

**Inputs:** `data/raw/` (enhancers, CRISPRi coefficients), stored predictions in
`data/processed/`.
**Outputs:** enhancer-level main effects and interactions for notebook 02, figures,
and `results/summaries/enhancer_summary.txt`.

## Setup

In [ ]:
import ast
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from alphagenome.data import genome
from alphagenome.models import dna_client
from alphagenome_research.model.variant_scoring.gene_mask_extractor import (
    GeneMaskExtractor, GeneMaskType, GeneQueryType
)
from pyfaidx import Fasta
from tangermeme.ersatz import dinucleotide_shuffle
from tangermeme.utils import one_hot_encode

# Make the package in ../src importable without installing it.
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import element_epistasis.enhancer_analysis as ea
import element_epistasis.enhancer_plots as ep
import element_epistasis.tf_cooperativity as tc
from element_epistasis import paths
from element_epistasis.GLM_fit import (
    fit_all_genes, fit_interaction_glm, fit_interaction_glm_atac_all_genes,
    plot_coef_forest, plot_coef_forest_grid,
)
from element_epistasis.pipeline import run_all

paths.ensure_output_dirs()

### Inputs

Enhancers are listed once per target gene: an element assigned to two genes
(`GENE1_enhA:GENE2_enhB`) is analysed separately in each gene's window. CRISPRi
coefficients measured in several datasets are averaged.

In [ ]:
# Enhancers, one row per enhancer and target gene
def extract_genes(name_string):
    return [part.split('_')[0] for part in name_string.split(':')]
def extract_enhancers(name_string):
    return [part for part in name_string.split(':')]

enhancers = pd.read_csv(paths.RAW / "enhancers.bed", sep="\t", header=None,
                        names=["chrom", "start", "end", "name"])
enhancers['target_gene'] = enhancers['name'].apply(extract_genes)
enhancers['enhancer_name'] = enhancers['name'].apply(extract_enhancers)
enhancers = enhancers.explode(['target_gene', 'enhancer_name'], ignore_index=True)

# Measured CRISPRi coefficients, averaged over datasets
inter_betas = pd.read_csv(paths.RAW / "pair_interaction_adj.csv")
inter_betas = inter_betas.groupby(["enhA", "enhB"], observed=True).agg({"interaction_beta": "mean"}).reset_index()

single_betas = pd.read_csv(paths.RAW / "enhancer_betas.csv")
single_betas = single_betas.groupby("enhancer").agg({"beta": "mean"}).reset_index()
single_betas = single_betas.rename(columns={"beta": "interaction_beta"})

# GENCODE v46 protein-coding annotation (the annotation AlphaGenome uses)
gtf_df = paths.load_gtf()
gtf_df = gtf_df[gtf_df.gene_type == "protein_coding"]

enhancers

In [ ]:
CONTEXT_LENGTH = dna_client.SEQUENCE_LENGTH_1MB   # AlphaGenome input window, 1,048,576 bp
FIRST_NON_N_BASE = 60_000                          # windows start no earlier than this (leading N's in hg38)
N_SHUFFLES = 10                                    # dinucleotide shuffles per scrambled enhancer set
SHUFFLE_SEED = 42                                  # seed of the dinucleotide shuffles
API_MAX_RETRIES = 5                                # prediction attempts per gene before stopping
API_MAX_WORKERS = 6                                # parallel API requests, reduced after errors
API_RETRY_WAIT_S = 15                              # wait before a retry, multiplied by the attempt number
GLM_FAMILY = "gamma"                               # response distribution of the interaction GLM
MAX_INTERACTION_ORDER = 2                          # highest interaction order of the main GLM
EXAMPLE_GENE = "ANXA1"                             # gene used to compare GLM families and readouts
FIGURE_GENES = ["PRKAR2B", "DDIT4"]                # genes shown as worked examples

## Prediction windows

Each gene gets one window that contains its gene body and all of its enhancers,
so that every combination of perturbations is predicted in the same context.
The window is centred on that span and clipped at the start of the chromosome.
Enhancer coordinates are then expressed relative to the window.

In [ ]:
def get_gene_coord(gene):
    ret = gtf_df[(gtf_df.Feature == "gene") & (gtf_df.gene_name == gene)]
    if ret.empty:
        return None, None
    return ret.Start.values[0], ret.End.values[0]

enhancers['gene_start'], enhancers['gene_end'] = zip(*enhancers['target_gene'].apply(get_gene_coord))

# Span of gene body and all enhancers, per gene
enhancers['iv_min_start'] = (
    enhancers[['start', 'gene_start']]
    .min(axis=1)
    .groupby(enhancers['target_gene'])
    .transform('min')
)
enhancers['iv_max_end'] = (
    enhancers[['end', 'gene_end']]
    .max(axis=1)
    .groupby(enhancers['target_gene'])
    .transform('max')
)

assert (enhancers.iv_max_end - enhancers.iv_min_start <= CONTEXT_LENGTH).all(), "All Enhancer gene pairs must be in the prediction input interval"

plt.hist(enhancers.iv_max_end - enhancers.iv_min_start, bins=len(enhancers))
plt.xlim(0, CONTEXT_LENGTH)
plt.xlabel("Span of gene and enhancers (bp)")
plt.show()

In [ ]:
iv_center = enhancers.iv_min_start + (enhancers.iv_max_end - enhancers.iv_min_start) // 2
iv_start = iv_center - CONTEXT_LENGTH // 2
iv_start[iv_start < FIRST_NON_N_BASE] = FIRST_NON_N_BASE
iv_end = iv_start + CONTEXT_LENGTH
assert (iv_end - iv_start == CONTEXT_LENGTH).all()

enhancers["iv_start"] = iv_start
enhancers["iv_end"] = iv_end
enhancers["interval"] = [genome.Interval(chromosome=row.chrom, start=row.iv_start, end=row.iv_end).resize(CONTEXT_LENGTH)
                         for row in enhancers.itertuples()]

# Enhancer coordinates relative to the window
enhancers['start'] = enhancers['start'] - enhancers['iv_start']
enhancers['end'] = enhancers['end'] - enhancers['iv_start']
enhancers['seq_idx'] = pd.factorize(enhancers['target_gene'])[0]
enhancers

## In silico knockout predictions

The cells in this section rerun AlphaGenome for every enhancer combination. They
need an AlphaGenome API key (`ALPHAGENOME_API_KEY` or `~/.alphagenome_api_key`),
the hg38 FASTA in `data/reference/` and several hours of API time.

**To skip them**, run only the next cell, which loads the stored predictions,
and continue at *Interaction GLM*. The cells after it are tagged `prediction`
and regenerate the same two tables from scratch.

In [ ]:
# Skip the regeneration: load the stored predictions
scram_effects = pd.read_parquet(paths.PROCESSED / "enhancer_insilico_KO.parquet", engine='pyarrow')
scram_effects_agg = pd.read_parquet(paths.PROCESSED / "enhancer_insilico_KO_agg.parquet", engine='pyarrow')
scram_effects.head()

### Model and K562 tracks

Predictions are read from the K562 accessibility and RNA tracks.

In [ ]:
model = dna_client.create(paths.alphagenome_api_key())

# filter here to alphagenome tracks relvant to you
track_metadata = model.output_metadata(dna_client.Organism.HOMO_SAPIENS).concatenate()
K562_metadata = track_metadata[(track_metadata["biosample_name"].str.contains("K562", case=False, na=False)) &
                               ((track_metadata["output_type"] == dna_client.OutputType.ATAC) |
                                (track_metadata["output_type"] == dna_client.OutputType.CHIP_TF))
                               ].dropna(how="all", axis=1)

K562_curie = K562_metadata.ontology_curie.unique().tolist()
print("curie:", K562_curie)
print("samples:", K562_metadata.biosample_name.unique().tolist())

# Windows must not reach into the N-padded chromosome ends
chrom_size = pd.read_csv(paths.CHROM_SIZES, sep="\t", names=["chrom", "end"])
chrom_size_dict = dict(zip(chrom_size["chrom"], chrom_size["end"]))
assert len([idx for idx, row in enhancers.iterrows()
            if row.iv_end > chrom_size_dict[row.chrom] - FIRST_NON_N_BASE]) == 0, "Some interval reach chrom ends."

### Shuffled donor sequences

A scrambled enhancer is replaced by the same coordinates taken from a
dinucleotide-shuffled copy of the whole window, so local base composition is
kept while motifs are destroyed. N bases are removed before shuffling and the
shuffled sequence is padded back to full length.

In [ ]:
hg38 = Fasta(str(paths.HG38_FASTA))
enhancers["seq"] = [str(hg38[iv.chromosome][iv.start:iv.end]) for iv in enhancers.interval]

seq_1hot = torch.stack([one_hot_encode(seq) for seq in enhancers.seq])

# Drop positions that are N in any window before shuffling
valid_cols_mask = (seq_1hot.sum(dim=1) == 1).all(dim=0)
filtered_1hot = seq_1hot[:, :, valid_cols_mask]
print("Filtered shape:", filtered_1hot.shape)

dishuff = dinucleotide_shuffle(filtered_1hot, n=N_SHUFFLES, random_state=SHUFFLE_SEED)
dishuff_pad = dishuff[:, :, :, :CONTEXT_LENGTH - filtered_1hot.shape[2]]
dishuff = torch.cat([dishuff, dishuff_pad], dim=3)
dishuff.shape

### Scramble combinations

For each gene, every subset of its enhancers is scrambled: the reference
sequence once, and each non-empty subset once per shuffle.

In [ ]:
_IDX_TO_NUC = np.array(["A", "C", "G", "T"])


def onehot_to_seq(onehot: torch.Tensor) -> str:
    # Convert a one-hot tensor of shape [4, L] to an uppercase nucleotide string.
    indices = onehot.argmax(dim=0).numpy()
    return "".join(_IDX_TO_NUC[indices])


def splice_scramble(
    original_seq: str,
    scramble_windows: list[tuple[int, int]],
    donor_seq: str,
) -> str:
    # Replace every (start, end) window of original_seq with the same window of donor_seq.
    seq = list(original_seq)
    for start, end in scramble_windows:
        seq[start:end] = list(donor_seq[start:end])
    return "".join(seq)


def generate_combination_df(
    motif_df: pd.DataFrame,
    dishuff: torch.Tensor,
    enhancer_col: str = "name",
    seq_col: str = "seq",
    seq_idx_col: str = "seq_idx",
    start_col: str = "start_seq",
    end_col: str = "end_seq",
    seqlet_name_col: str = "example_idx",
) -> pd.DataFrame:
    # One row per scramble combination and shuffle (the reference once, without shuffle).
    n_shuffles = dishuff.shape[1]
    rows = []

    for enhancer, grp in motif_df.groupby(enhancer_col):
        grp = grp.reset_index(drop=True)

        original_seq = grp[seq_col].iloc[0].upper()
        seq_idx      = int(grp[seq_idx_col].iloc[0])

        grp_sorted = grp.sort_values(start_col).reset_index(drop=True)
        n_seqlets  = len(grp_sorted)

        enhancer_shuffles = dishuff[seq_idx]
        shuffle_seqs = [onehot_to_seq(enhancer_shuffles[i]) for i in range(n_shuffles)]

        seqlet_labels  = grp_sorted[seqlet_name_col].tolist()
        seqlet_windows = list(zip(grp_sorted[start_col].tolist(), grp_sorted[end_col].tolist()))

        # bit i of combo_id set -> unit i is scrambled
        for combo_id in range(2 ** n_seqlets):
            bits = [(combo_id >> i) & 1 for i in range(n_seqlets)]

            scrambled_idx   = [i for i, b in enumerate(bits) if b]
            unscrambled_idx = [i for i, b in enumerate(bits) if not b]

            scrambled_labels   = [seqlet_labels[i]  for i in scrambled_idx]
            unscrambled_labels = [seqlet_labels[i]  for i in unscrambled_idx]
            scrambled_windows  = [seqlet_windows[i] for i in scrambled_idx]

            combo_label = (
                f"scr[{','.join(str(s) for s in scrambled_labels)}]"
                f"_unscr[{','.join(str(s) for s in unscrambled_labels)}]"
            )

            if combo_id == 0:
                rows.append(dict(
                    name=enhancer, seq_idx=seq_idx, combo_id=combo_id,
                    combo_label=combo_label, scrambled_seqlets=scrambled_labels,
                    unscrambled_seqlets=unscrambled_labels, n_scrambled=0,
                    shuffle_idx=None, sequence=original_seq,
                ))
            else:
                for shuf_i, donor in enumerate(shuffle_seqs):
                    rows.append(dict(
                        name=enhancer, seq_idx=seq_idx, combo_id=combo_id,
                        combo_label=combo_label, scrambled_seqlets=scrambled_labels,
                        unscrambled_seqlets=unscrambled_labels,
                        n_scrambled=len(scrambled_idx), shuffle_idx=shuf_i,
                        sequence=splice_scramble(original_seq, scrambled_windows, donor),
                    ))

    result = pd.DataFrame(rows)
    col_order = [
        "name", "seq_idx", "combo_id", "combo_label",
        "n_scrambled", "scrambled_seqlets", "unscrambled_seqlets",
        "shuffle_idx", "sequence",
    ]
    return result[col_order]

In [ ]:
out_df = generate_combination_df(enhancers, dishuff, seq_col="seq", seq_idx_col="seq_idx",
                                 start_col="start", end_col="end", enhancer_col="target_gene",
                                 seqlet_name_col="enhancer_name")

# The first enhancer's left boundary in the reference and the first two scrambles
first_start = int(enhancers['start'].iloc[0])
for seq in out_df.sequence.iloc[:3]:
    print(f"{seq[first_start - 10:first_start]} {seq[first_start:first_start + 10]}")

print(f"Total rows: {len(out_df)}   genes: {out_df['name'].nunique()}")
seq_lens = out_df.groupby("name")["sequence"].apply(lambda s: s.str.len().unique())
assert all(len(v) == 1 for v in seq_lens), "Sequence length mismatch!"

### Predict expression and accessibility

Expression is the mean predicted RNA coverage over the target gene's exons, on
its strand; accessibility is the predicted ATAC signal summed over each enhancer
of the gene. Shuffles are kept as replicates (`scram_effects`) and also averaged
(`scram_effects_agg`).

In [ ]:
exon_extractor = GeneMaskExtractor(
    gtf=gtf_df,
    gene_mask_type=GeneMaskType.EXONS,
    gene_query_type=GeneQueryType.INTERVAL_CONTAINED,
    filter_protein_coding=False
)


def mean_of_lists(series):
    # Element-wise mean across a series of lists.
    parsed = [ast.literal_eval(v) if isinstance(v, str) else v for v in series]
    return np.array(parsed).mean(axis=0).tolist()


def sd_of_lists(series):
    # Element-wise standard deviation across a series of lists.
    parsed = [ast.literal_eval(v) if isinstance(v, str) else v for v in series]
    return np.array(parsed).std(axis=0).tolist()


df_list, df_list_agg = [], []

for name in out_df.name.unique():
    print(f"\nAnalysing enhancer: {name}")
    enh_scram = out_df[out_df.name == name]
    edata = enhancers[enhancers.target_gene == name]

    input_interval = edata.interval.values[0]
    model_inputs = enh_scram.sequence.to_list()

    # Retry with fewer workers when the API errors, instead of aborting the loop
    out_pred = None
    current_workers = API_MAX_WORKERS
    for attempt in range(API_MAX_RETRIES):
        try:
            out_pred = model.predict_sequences(
                sequences=model_inputs,
                requested_outputs=[dna_client.OutputType["ATAC"], dna_client.OutputType["RNA_SEQ"]],
                ontology_terms=K562_curie,
                max_workers=current_workers,
                intervals=[input_interval] * len(model_inputs)
            )
            break
        except Exception as e:
            print(f"     [!] API Error on attempt {attempt + 1}: {e}")
            if attempt < API_MAX_RETRIES - 1:
                current_workers = current_workers - 2 if current_workers > 2 else 1
                wait_time = API_RETRY_WAIT_S * (attempt + 1)
                print(f"     [*] max_workers={current_workers}, retrying in {wait_time}s")
                time.sleep(wait_time)
            else:
                raise e

    # Exon mask of the target gene, once per window
    gene_mask, annot = exon_extractor.extract(interval=out_pred[0].rna_seq.interval)
    is_target = annot.gene_name == name
    annot = annot.loc[is_target]
    gene_mask = gene_mask[:, is_target]

    RNA_counts, ATAC_counts, enhs = [], [], []
    for seq_out in out_pred:
        # ATAC summed over each enhancer of the gene
        enh_name, enh_counts = [], []
        for e in edata.itertuples():
            enh_name.append(e.enhancer_name)
            mask = np.zeros(CONTEXT_LENGTH, dtype=bool)
            mask[e.start:e.end] = True
            enh_counts.append(seq_out.atac.values.mean(axis=1)[mask].sum())
        enhs.append(enh_name)
        ATAC_counts.append(enh_counts)

        # RNA averaged over the exons of the target gene, matching strand
        strand = annot.strand.values[0]
        stranded_track = seq_out.rna_seq.metadata.strand.isin([strand, "."])
        RNA_counts.append(seq_out.rna_seq.values[:, stranded_track].mean(axis=1)[gene_mask[:, 0]].mean())

    del out_pred
    enh_scram["RNA_counts"] = RNA_counts
    enh_scram["ATAC_counts"] = ATAC_counts
    enh_scram["target_enhancer"] = enhs

    enh_scram_agg = (
        enh_scram
        .groupby(["name", "seq_idx", "combo_id", "combo_label", "n_scrambled"], observed=True)
        .agg(
            ATAC_counts         = ("ATAC_counts",         mean_of_lists),
            ATAC_counts_sd      = ("ATAC_counts",         sd_of_lists),
            RNA_counts          = ("RNA_counts",          "mean"),
            RNA_counts_sd       = ("RNA_counts",          "std"),
            target_enhancer     = ("target_enhancer",     "first"),
            scrambled_seqlets   = ("scrambled_seqlets",   "first"),
            unscrambled_seqlets = ("unscrambled_seqlets", "first"),
        )
        .reset_index()
    )
    df_list.append(enh_scram)
    df_list_agg.append(enh_scram_agg)
    del enh_scram, enh_scram_agg

In [ ]:
scram_effects = pd.concat(df_list).reset_index(drop=True)
scram_effects_agg = pd.concat(df_list_agg).reset_index(drop=True)

# The 1 Mb input sequences are only needed for prediction and are not stored
scram_effects.drop(columns="sequence").to_parquet(paths.PROCESSED / "enhancer_insilico_KO.parquet", engine='pyarrow', index=False)
scram_effects_agg.to_parquet(paths.PROCESSED / "enhancer_insilico_KO_agg.parquet", engine='pyarrow', index=False)

## Interaction GLM

For each gene, predicted expression of every shuffle replicate is modelled as

$$\log E[y] = \beta_0 + \sum_i \beta_i\,\mathbb{1}[i\ \text{scrambled}] + \sum_{i<j} \beta_{ij}\,\mathbb{1}[i, j\ \text{scrambled}]$$

$\beta_i$ is the log fold change from losing enhancer $i$, and $\beta_{ij}$ is how
far the pair departs from the product of the two single effects.

### Choice of response distribution

Predictions are continuous, so a negative binomial would need rounded values.
Both families are compared on one gene.

In [ ]:
for fam in ["nb", "gamma"]:
    res = fit_interaction_glm(scram_effects, EXAMPLE_GENE, family=fam,
                              max_order=MAX_INTERACTION_ORDER, verbose=True)
    fig, ax = plt.subplots(figsize=(6, 4))
    plot_coef_forest(res, ax=ax)
    plt.tight_layout(); plt.show()

Rounding for the negative binomial distorts the continuous predictions, so the
Gamma family is used.

### Fit all genes

Adding third-order terms leaves the pairwise estimates unchanged and only
widens their standard errors, so the second-order model is used downstream.

In [ ]:
interactions_gm = fit_all_genes(scram_effects, family=GLM_FAMILY, max_order=MAX_INTERACTION_ORDER)
interactions_gm_df = pd.concat([fit['coef_table'].assign(gene=gene) for gene, fit in interactions_gm.items()])

interactions_gm_3 = fit_all_genes(scram_effects, family=GLM_FAMILY, max_order=3)
interactions_gm_3_df = pd.concat([fit['coef_table'].assign(gene=gene) for gene, fit in interactions_gm_3.items()])

pd.merge(interactions_gm_3_df, interactions_gm_df, on=["gene", "family", "term"], how="inner", suffixes=("_3", "_2"))

## Comparison with the CRISPRi screen

Main effects are matched to measured single-enhancer effects and interaction
terms to measured pair interactions, by enhancer name.

In [ ]:
in_silico_single = interactions_gm_df[~interactions_gm_df['term'].str.contains(":|intercept", na=False)].copy()
in_silico_single["enhancer"] = in_silico_single["gene"] + "_" + in_silico_single["term"]

in_silico_inter = interactions_gm_df[interactions_gm_df['term'].str.contains(":", na=False)].copy()
in_silico_inter[['enhA', 'enhB']] = [(g + "_" + a, g + "_" + b)
                                     for g, (a, b) in zip(in_silico_inter['gene'], in_silico_inter['term'].str.split(":"))]

single_comp = pd.merge(in_silico_single, single_betas, on="enhancer", how="inner")
pair_comp = pd.merge(in_silico_inter, inter_betas, on=["enhA", "enhB"], how="inner")
print(f"single enhancers matched: {len(single_comp)}   enhancer pairs matched: {len(pair_comp)}")

## Accessibility GLM per enhancer

The same model is fitted to the predicted accessibility of each enhancer of a
gene, giving the effect of every perturbation on every element. Across loci,
each pair keeps its strongest interaction.

In [ ]:
atac_results = fit_interaction_glm_atac_all_genes(scram_effects, family=GLM_FAMILY, max_order=MAX_INTERACTION_ORDER)
rna_result = fit_interaction_glm(scram_effects, gene=EXAMPLE_GENE, family=GLM_FAMILY, max_order=MAX_INTERACTION_ORDER)

fig = plot_coef_forest_grid(rna_result=rna_result, atac_result=atac_results[EXAMPLE_GENE],
                            crispr_single=single_betas, crispr_inter=inter_betas)
plt.show()

In [ ]:
atac_results_df = pd.concat([fit['coef_table'].assign(gene=gene) for gene, fit in atac_results.items()])
atac_results_df = atac_results_df[atac_results_df['term'].str.contains(":", na=False)]
atac_results_df[['enhA', 'enhB']] = [(g + "_" + a, g + "_" + b)
                                     for g, (a, b) in zip(atac_results_df['gene'], atac_results_df['term'].str.split(":"))]

def absmax(series: pd.Series) -> float:
    return series.loc[series.abs().idxmax()]

atac_results_df = atac_results_df.groupby(["gene", "enhA", "enhB"], observed=True).agg({"coef": absmax, "pval": "min"}).reset_index()
atac_comp = pd.merge(atac_results_df, inter_betas, on=["enhA", "enhB"], how="inner")
atac_comp.head(3)

## Effect tables, cooperativity and validation

Coefficients are split into main effects and interactions with
Benjamini-Hochberg correction, annotated with the main effects and genomic
distance of each pair, and saved for notebook 02. Gene-level cooperativity
compares the effect of scrambling all enhancers with the sum of the single
effects.

In [ ]:
mains, inters = ea.split_coef_table(interactions_gm_df)
mains = tc.add_bh_correction(mains)

inters = tc.add_bh_correction(inters)
inters = ea.attach_main_effects(inters, interactions_gm_df)   # needs the unsplit table (intercept)
inters = ea.annotate_pair_distance(inters, enhancers)

# Inputs of the cross-enhancer motif analysis in notebook 02
mains.to_csv(paths.PROCESSED / "enhancer_insilico_single_KO_main_effects.csv", header=True, index=False)
inters.to_csv(paths.PROCESSED / "enhancer_insilico_KO_inters.csv", header=True, index=False)

In [ ]:
atac_inters = ea.normalise_interaction_table(atac_results_df)
atac_inters = tc.add_bh_correction(atac_inters)

coop_rna = ea.build_coop_table(scram_effects, n_scrambled_col="n_scrambled", value_col="RNA_counts")
epi_rna = ea.build_pairwise_epistasis(scram_effects, n_scrambled_col="n_scrambled", value_col="RNA_counts")

summaries = {
    "single enhancers": ea.summarise_validation(single_comp, insilico_p_col="pval"),
    "enhancer pairs": ea.summarise_validation(pair_comp, insilico_p_col="pval"),
}

fig = ep.make_main_figure(inters, coop_rna, single_comp, pair_comp, summaries)
plt.show()

Restricting the pair comparison to pairs significant in silico does not improve
agreement with the screen.

In [ ]:
fig, ax = plt.subplots(1, 1)
sig_pair_comp = tc.add_bh_correction(pair_comp)
ep.plot_insilico_vs_measured(sig_pair_comp[sig_pair_comp.significant], ax=ax, level="pair")
plt.show()

## Model family and interaction order

Gamma and negative binomial models with interactions up to order four, compared
by the mean standard error of their coefficients.

In [ ]:
interactions_gm_4 = fit_all_genes(scram_effects, family="gamma", max_order=4)
interactions_nb = fit_all_genes(scram_effects, family="nb", max_order=2)
interactions_nb_3 = fit_all_genes(scram_effects, family="nb", max_order=3)
interactions_nb_4 = fit_all_genes(scram_effects, family="nb", max_order=4)

comparison = ea.build_model_comparison({
    "gamma, order 2": interactions_gm,
    "gamma, order 3": interactions_gm_3,
    "gamma, order 4": interactions_gm_4,
    "NB, order 2":    interactions_nb,
    "NB, order 3":    interactions_nb_3,
    "NB, order 4":    interactions_nb_4,
}, verbose=False)

fig = ep.make_supplementary_figure(inters, inters, atac_inters, comparison=comparison, figsize=(10, 10))
plt.show()

## Accessibility cooperativity and crosstalk

Cooperativity on accessibility is scored once per gene (averaged over its
enhancers) and once per enhancer locus. The interdependency maps show how
scrambling one enhancer changes the accessibility of the others (*trans*
effects), and whether that crosstalk explains cooperativity.

In [ ]:
# One accessibility value per gene
summed = ea.aggregate_locus_values(scram_effects, "ATAC_counts", how="mean")
coop_atac_sum = ea.build_coop_table(summed, n_scrambled_col="n_scrambled", value_col="ATAC_counts")

# One accessibility value per enhancer locus
long = ea.expand_locus_values(scram_effects, "ATAC_counts")
coop_atac_loc = ea.build_coop_table(long, name_col="locus", n_scrambled_col="n_scrambled", value_col="ATAC_counts")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
ep.plot_additivity(coop_atac_sum, ax=axes[0], readout="ATAC, mean")
ep.plot_additivity(coop_atac_loc, ax=axes[1], readout="ATAC, per locus")
ep.plot_coop_agreement(coop_atac_sum, coop_atac_loc, ax=axes[2])
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(18, 18))
ep.plot_global_interdependency(atac_results, ax=ax, show_only_sig=True,
                               hatch_nonsignificant=True, clip_quantile=0.3,
                               ylabel_pad=55.0, gene_label_x=-0.056)
plt.show()

In [ ]:
interdependency_atac = ea.build_interdependency(long, value_col="ATAC_counts")

trans_gene = ea.summarise_trans_effects(interdependency_atac, by="gene")
trans_locus = ea.summarise_trans_effects(interdependency_atac, by="perturbed")

pair_trans = ea.summarise_pair_trans(interdependency_atac)
pair_comp_2 = ea.join_pair_trans(pair_comp, pair_trans)

ep.make_mechanism_figure(trans_gene, trans_locus, coop_atac_sum, coop_atac_loc, coop_rna,
                         pair_comp=pair_comp_2, figsize=(10, 10))
plt.show()

## Example genes

Coefficients, pairwise interactions and every scramble combination for single genes.

In [ ]:
for gene in FIGURE_GENES:
    fig = ep.make_gene_figure(gene, interactions_gm[gene], epi_rna, scram_effects_agg)
    plt.show()

## Text summary

All enhancer-level statistics, with sample sizes and p-values, in
`results/summaries/enhancer_summary.txt`.

In [ ]:
tables = run_all(
    enh_coef=interactions_gm_df,
    enh_atac_coef=atac_results_df,
    enh_scram=scram_effects,
    enhancers=enhancers,
    single_comp=single_comp,
    pair_comp=pair_comp,
    model_results={"gamma, order 2": interactions_gm,
                   "gamma, order 3": interactions_gm_3},
    summary_path=paths.SUMMARIES / "enhancer_summary.txt",
)